# Data Preparation for geospatial analysis

##### **Step 1: Load the datasets**
##### **Step 2: Address the multiple cities in the city column.**
##### **Step 3: Clean the city names for better matching.**
##### **Step 4: Perform a self-join to find corrosponding recinded notice of advisory.**
##### **Step 5: Perform cross join and fuzzy logic with PWS dataset to get the city (or location), population and other information from the PWS dataset for better geospatial analysis.**

In [1]:
import pandas as pd
import logging

logging.basicConfig(
    level=logging.INFO,
    format="[Info] %(message)s",
    force=True
)

# Loading JSON file
df = pd.read_json("merged_json_output/merged_output.json")

##### **Indentifying multiple cities in city column and working on those cities**
##### **Purpose:** Trying to get individual record for each cities.

In [2]:
# check the city column first, its datatype
df['city'].apply(type).value_counts()

city
<class 'str'>         723
<class 'list'>         52
<class 'NoneType'>      1
Name: count, dtype: int64

In [3]:
multiple_city_count = df['city'].apply(lambda x: isinstance(x, list)).sum()
logging.info(f"Total rows before exploding multiple cities: {multiple_city_count}")

[Info] Total rows before exploding multiple cities: 52


In [4]:
multiple_city_rows = df[df['city'].apply(
    lambda x: isinstance(x, list) and len(x) > 1
)]
multiple_city_rows.head(2)

,url,title,combined_context,extracted_entities_advisory_reason,posted_on,issued_date,rescinded_date,year,city,city_type,county
26,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,Boil Water Advisory Rescinded for Sunset Valle...,The advisory was issued because of a loss of p...,"[loss of pressure, loss of chlorine residuals]","Posted on December 06, 2025",None,12/06/2025,2025.0,"[Sunset Valley MHP, Hays, KS]",unstructured,Ellis County
28,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,Boil Water Advisory Rescinded for Tuttle CC Mo...,The advisory was issued because of a mechanica...,[Failure to maintain required chlorine residua...,"Posted on December 04, 2025",None,12/04/2025,2025.0,"[Tuttle CC Mobile Home Park, LLC]",unstructured,Riley County


##### **Lets prepare the city column for the cleaning purpose because of the variation of formats in the list**
we got list in the following forms : ['City/PWS A', 'City/PWS B'], ['City/PWS ,city, county, state code'] etc

In [5]:
import pandas as pd
import ast
import re


# this function parse the input value to list if it is a stringified list, 
# or return a list with the single value if it's just a string
def parse_city_field(val):
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        # Parsing stringified lists like "['City A', 'City B']"
        if val.startswith('['):
            try:
                return ast.literal_eval(val)
            except:
                pass
    return [val]  # wrapping single string in list


def split_city_list(items):
    state_codes = {'KS', 'MO', 'TX', 'OK', 'NE', 'CO'}
    business_suffixes = {'INC', 'INC.', 'LLC', 'LLC.', 'LTD', 'LTD.', 'CORP', 'CORP.', 'CO', 'CO.', 'MHP'}
    
    if any(str(item).strip().upper() in state_codes for item in items):
        # It's a full address — join back into one string
        # If found any state code, it is assumed to be a full address and make it one whole string if seperated by comma
        return [', '.join(str(i).strip() for i in items)]
    
    elif any(str(item).strip().upper() in business_suffixes for item in items):
        # It's a business name — join back into one string
        # If found any business suffix, it is assumed to be a business name and make it one whole string if seperated by comma
        return [', '.join(str(i).strip() for i in items)]
    
    else:
        # Multiple separate cities like "City A, City B" — keep as list
        return [str(i).strip() for i in items]



In [6]:
# applying the above function on city column
multiple_city_rows['city_parsed'] = multiple_city_rows['city'].apply(parse_city_field)
# multiple_city_rows['city_parsed'] = multiple_city_rows['city_parsed'].apply(split_city_list)


C:\Users\aarun\AppData\Local\Temp\ipykernel_88548\2089149786.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  multiple_city_rows['city_parsed'] = multiple_city_rows['city'].apply(parse_city_field)


In [7]:
# Apply the actual splitting function to the parsed city lists
multiple_city_rows['city_parsed'] = multiple_city_rows['city_parsed'].apply(split_city_list)

C:\Users\aarun\AppData\Local\Temp\ipykernel_88548\3855409236.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  multiple_city_rows['city_parsed'] = multiple_city_rows['city_parsed'].apply(split_city_list)


##### **Now we assume that the city list is normalized and sparced as needed to explode the rows into myltiple rows respectively to its city counts in the list**

In [8]:
# Now exploding into individual rows
multiple_city_rows_parsed = multiple_city_rows.copy().explode('city_parsed').drop(columns='city').rename(columns={'city_parsed': 'city'})
multiple_city_rows_parsed = multiple_city_rows_parsed.reset_index(drop=True)

In [9]:
# saving it to an excel file to check the results
# multiple_city_rows_parsed[['url','city', 'county']].to_excel("multiple_city_rows_parsed_exploded.xlsx", index=False)

In [10]:
multiple_city_rows_parsed[['city', 'county']].head(5)

,city,county
0,"Sunset Valley MHP, Hays, KS",Ellis County
1,"Tuttle CC Mobile Home Park, LLC",Riley County
2,"Sunset Valley MHP, Hays, KS",Ellis County
3,"Tuttle CC Mobile Home Park, LLC",Riley County
4,"Sundowner, Inc, AKA Sundowner West Mobile Home...",Saline County


In [11]:
len(multiple_city_rows_parsed)
logging.info(f"Total rows after exploding multiple cities: {len(multiple_city_rows_parsed)}")


[Info] Total rows after exploding multiple cities: 98


#### **Cases of only one location entity or city/PWS entity**

In [12]:
# selecting only where city is a single string (not a list of cities)
single_city_df = df[
    ~df['city'].apply(lambda x: isinstance(x, list) and len(x) > 1)
]

In [13]:
single_city_df.shape
logging.info(f"Total rows with single city entries: {single_city_df.shape[0]}")

[Info] Total rows with single city entries: 724


In [14]:
# verifying that there are no more list entries in the city column
single_city_df['city'].apply(
    lambda x: isinstance(x, list) and len(x) > 1
).sum()
logging.info(f"Total rows with list entries in city column after filtering: {single_city_df['city'].apply(lambda x: isinstance(x, list) and len(x) > 1).sum()}")

[Info] Total rows with list entries in city column after filtering: 0


In [15]:
single_city_df['city'].apply(type).value_counts()   

city
<class 'str'>         723
<class 'NoneType'>      1
Name: count, dtype: int64

In [16]:
multiple_city_rows_parsed['city'].apply(type).value_counts()

city
<class 'str'>    98
Name: count, dtype: int64

##### **Since we have addressed the multiple city/pws entity in city columns, Now cleaning that column is necessary**

In [17]:
# combining single city and multiple city dataframe
city_processed_combined_df = pd.concat([single_city_df, multiple_city_rows_parsed], ignore_index=True)
logging.info(f"Total rows in combined dataframe: {city_processed_combined_df.shape[0]}")

[Info] Total rows in combined dataframe: 822


In [18]:
city_processed_combined_df[['url', 'city', 'county']].to_excel("city_processed_combined.xlsx", index=False)

In [19]:
combined_df = city_processed_combined_df.copy()

##### **Cleaning function**

In [20]:
import re

def clean_for_matching(text):
    if pd.isna(text):
        return text
    text = str(text).lower()
    
    # Addressing the most common variations and abbreviations in the city names.
    # Abbreviations (order matters!! longer phrases first)
    abbreviations = {
        r'rural water district'   : 'rwd',
        r'mobile home park'       : 'mhp',
        r'mobile home court'      : 'mhc',
        r'rural water supply'     : 'rws',
        r'public water supply'    : 'pws',
        r'water district'         : 'wd',
        r'county'                 : 'co',
        r'association'            : 'assoc'
    }
    for pattern, replacement in abbreviations.items():
        text = re.sub(pattern, replacement, text)
    
    # Removing "no." or "no" before numbers (rwd no. 1 → rwd 1)
    text = re.sub(r'\bno\.?\s*(?=\d)', '', text)
    
    # Remove filler words/phrases, put longer phrase first to avoid partial matches
    fillers = [
        r'a portion of the',
        r'a portion of',
        r'portions? of',        
        r'located',
        r'serving',
        r'the city of',
        r'city of',
        r'town of',
    ]
    for filler in fillers:
        text = re.sub(filler, '', text)
    
    # --- Symbol cleanup ---
    text = re.sub(r'#', '', text)       # rwd#1 → rwd1
    text = re.sub(r'\.', '', text)      # remove periods
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

combined_df['city_cleaned'] = combined_df['city'].apply(clean_for_matching)


##### **Now after cleaning the city column, lets divide the dataframe based onthe issue date and recind date because we need to combine those into one record. So that each advisory have its own issued and recinded date**

##### **Lets check the datatype of date columns**

In [21]:
# checking the datatypes of combined_df
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 822 entries, 0 to 821
Data columns (total 12 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   url                                 822 non-null    object 
 1   title                               822 non-null    object 
 2   combined_context                    822 non-null    object 
 3   extracted_entities_advisory_reason  822 non-null    object 
 4   posted_on                           822 non-null    object 
 5   issued_date                         395 non-null    object 
 6   rescinded_date                      424 non-null    object 
 7   year                                819 non-null    float64
 8   city                                821 non-null    object 
 9   city_type                           822 non-null    object 
 10  county                              786 non-null    object 
 11  city_cleaned                        821 non-n

##### **change the date column to datetype**

In [22]:
# Convert to datetime
combined_df["issued_date"] = pd.to_datetime(combined_df["issued_date"], errors="coerce")
combined_df["rescinded_date"] = pd.to_datetime(combined_df["rescinded_date"], errors="coerce")

In [23]:
# Spliting issued and rescinded
issued_df = combined_df[combined_df["issued_date"].notna()].copy()
rescinded_df = combined_df[combined_df["rescinded_date"].notna()].copy()

In [24]:
logging.info(f"Total issued rows: {issued_df.shape[0]}")
logging.info(f"Total rescinded rows: {rescinded_df.shape[0]}")

[Info] Total issued rows: 395
[Info] Total rescinded rows: 424


In [25]:
# Self join on city + county , because some advisories have multiple cities, 
# we want to match on both city and county to avoid incorrect matches 
# (e.g. if there are two "Springfield" in different counties)
# So combining city and county into a unique key for matching
merged_issued_n_rescinded_df = issued_df.merge(
    rescinded_df,
    on=["city", "county"],
    how="left",
    suffixes=("_issued", "_rescinded")
)

In [26]:
merged_issued_n_rescinded_df.shape
logging.info(f"Total rows after merging issued and rescinded: {merged_issued_n_rescinded_df.shape[0]}")
logging.info(f"The count is higher because of the cross product during merging")

[Info] Total rows after merging issued and rescinded: 1032
[Info] The count is higher because of the cross product during merging


In [28]:
merged_issued_n_rescinded_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1032 entries, 0 to 1031
Data columns (total 22 columns):
 #   Column                                        Non-Null Count  Dtype         
---  ------                                        --------------  -----         
 0   url_issued                                    1032 non-null   object        
 1   title_issued                                  1032 non-null   object        
 2   combined_context_issued                       1032 non-null   object        
 3   extracted_entities_advisory_reason_issued     1032 non-null   object        
 4   posted_on_issued                              1032 non-null   object        
 5   issued_date_issued                            1032 non-null   datetime64[ns]
 6   rescinded_date_issued                         0 non-null      datetime64[ns]
 7   year_issued                                   1032 non-null   float64       
 8   city                                          1032 non-null   object

In [29]:
merged_issued_n_rescinded_df[['issued_date_issued', 'rescinded_date_rescinded']].head(5)

,issued_date_issued,rescinded_date_rescinded
0,2026-02-03,NaT
1,2026-02-02,2026-02-06
2,2026-02-02,2026-02-06
3,2026-02-02,2026-01-06
4,2026-02-02,2025-11-26


In [30]:
merged_issued_n_rescinded_df["duration_days"] = (
    merged_issued_n_rescinded_df["rescinded_date_rescinded"] - merged_issued_n_rescinded_df["issued_date_issued"]
).dt.days

logging.info(f"Duration days stats:\n{merged_issued_n_rescinded_df['duration_days'].describe()}")


[Info] Duration days stats:
count     953.000000
mean        6.925498
std       468.831454
min     -1827.000000
25%      -177.000000
50%         4.000000
75%       194.000000
max      1836.000000
Name: duration_days, dtype: float64


In [31]:
merged_issued_n_rescinded_df.to_excel("merged_issued_n_rescinded_df.xlsx", index=False)

In [32]:
# Keeping the rescind data that is most recent after the issued date of advisory for the same 
# city and county
merged_issued_n_rescinded_df = merged_issued_n_rescinded_df[
    merged_issued_n_rescinded_df["rescinded_date_rescinded"] > merged_issued_n_rescinded_df["issued_date_issued"]
]



In [33]:
merged_issued_n_rescinded_df.to_excel("merged_issued_n_rescinded_filtered_df.xlsx", index=False)

In [ ]:
# # Keep closest rescind per issued advisory
merged_issued_n_rescinded_filtered_df = (
    merged_issued_n_rescinded_df
    .sort_values("rescinded_date_rescinded")
    .groupby(["url_issued", "city", "county"], as_index=False)
    .first()
)


In [35]:
# After keeping the closest rescind, that are rescinded within 60 days after the issued date.
merged_issued_n_rescinded_filtered_df = merged_issued_n_rescinded_filtered_df[
    (merged_issued_n_rescinded_filtered_df["rescinded_date_rescinded"] - merged_issued_n_rescinded_filtered_df["issued_date_issued"]).dt.days <= 60
]

In [36]:
merged_issued_n_rescinded_filtered_df.to_excel("merged_issued_n_rescinded_filtered_closest_rescind_df.xlsx", index=False)

##### **Selecting the columns**

In [37]:
merged_issued_n_rescinded_filtered_df.columns

Index(['url_issued', 'city', 'county', 'title_issued',
       'combined_context_issued', 'extracted_entities_advisory_reason_issued',
       'posted_on_issued', 'issued_date_issued', 'rescinded_date_issued',
       'year_issued', 'city_type_issued', 'city_cleaned_issued',
       'url_rescinded', 'title_rescinded', 'combined_context_rescinded',
       'extracted_entities_advisory_reason_rescinded', 'posted_on_rescinded',
       'issued_date_rescinded', 'rescinded_date_rescinded', 'year_rescinded',
       'city_type_rescinded', 'city_cleaned_rescinded', 'duration_days'],
      dtype='object')

In [38]:
selected_cols = [
    'url_issued', 'city', 'county', 'title_issued',
    'combined_context_issued',
    'extracted_entities_advisory_reason_issued',
    'posted_on_issued', 'issued_date_issued',
    'year_issued', 'city_cleaned_issued',
    'url_rescinded', 'rescinded_date_rescinded',
    'year_rescinded', 'duration_days'
]

merged_issued_n_rescinded_filtered_df = (
    merged_issued_n_rescinded_filtered_df[selected_cols]
)
merged_issued_n_rescinded_filtered_df.head(2)
logging.info(f"Final dataset shape after filtering: {merged_issued_n_rescinded_filtered_df.shape}")

[Info] Final dataset shape after filtering: (279, 14)


##### **Step 5: Perform cross join with PWS dataset to get the city (or location), population and other information from the PWS dataset for better geospatial analysis.**

In [39]:
issued_rescinded_merged_df = merged_issued_n_rescinded_filtered_df.copy()

In [40]:
issued_rescinded_merged_df.columns

Index(['url_issued', 'city', 'county', 'title_issued',
       'combined_context_issued', 'extracted_entities_advisory_reason_issued',
       'posted_on_issued', 'issued_date_issued', 'year_issued',
       'city_cleaned_issued', 'url_rescinded', 'rescinded_date_rescinded',
       'year_rescinded', 'duration_days'],
      dtype='object')

In [41]:
pws_df = pd.read_excel("Data/SDWIS 2023 KS Water System Summary.xlsx", skiprows=4)
# pws_df = pd.read_csv("Data/SDWIS 2025 KS Water System Summary.csv")

In [42]:
pws_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 965 entries, 0 to 964
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   PWS ID                   965 non-null    object
 1   PWS Name                 965 non-null    object
 2   EPA Region Name          965 non-null    object
 3   Primacy Agency           965 non-null    object
 4   PWS Type                 965 non-null    object
 5   Population Served Count  965 non-null    int64 
 6   Cities Served            965 non-null    object
 7   Counties Served          965 non-null    object
 8   # of Facilities          965 non-null    int64 
 9   # of Violations          965 non-null    int64 
 10  # of Site Visits         965 non-null    int64 
dtypes: int64(4), object(7)
memory usage: 83.1+ KB


In [43]:
issued_rescinded_merged_df['city_cleaned_issued'] = issued_rescinded_merged_df['city_cleaned_issued'].str.strip().str.lower()
pws_df['PWS_clean'] = (
    pws_df['PWS Name']
    .str.split(',', n=1)
    .str[0]
    .str.strip()
    .str.lower()
)

##### **Applying Fuzzy Logic to maximize the matching between city/pws in BWA dataset and PWS dataset**

In [55]:
import re

def normalize_name(s):
    s = s.lower()
    s = re.sub(r"[^\w\s]", " ", s).strip()

    replacements = {
        r"\bassociation\b": "assn",
        r"\bassn\b": "assn",
        r"\bdistrict\b": "dist",
        r"\bcompany\b": "co",
        r"\bcounty\b": "co",
        r"\bmhc\b": "mobile home court",
        r"\bmhp\b": "mobile home park",
        r"\brwd\b": "rural water district"
    }

    for pattern, replacement in replacements.items():
        s = re.sub(pattern, replacement, s)

    return s

issued_rescinded_merged_df["city_norm"] = (
    issued_rescinded_merged_df["city_cleaned_issued"]
    .astype(str)
    .apply(normalize_name)
)

pws_df["pws_norm"] = (
    pws_df["PWS_clean"]
    .astype(str)
    .apply(normalize_name)
)

In [56]:

exact_matches = issued_rescinded_merged_df.merge(
    pws_df,
    left_on="city_norm",
    right_on="pws_norm",
    how="inner"
)

logging.info(f"Exact matches: {len(exact_matches)}")

[Info] Exact matches: 258


In [57]:
exact_matches.columns

Index(['url_issued', 'city', 'county', 'title_issued',
       'combined_context_issued', 'extracted_entities_advisory_reason_issued',
       'posted_on_issued', 'issued_date_issued', 'year_issued',
       'city_cleaned_issued', 'url_rescinded', 'rescinded_date_rescinded',
       'year_rescinded', 'duration_days', 'city_norm', 'PWS ID', 'PWS Name',
       'EPA Region Name', 'Primacy Agency', 'PWS Type',
       'Population Served Count', 'Cities Served', 'Counties Served',
       '# of Facilities', '# of Violations', '# of Site Visits', 'PWS_clean',
       'pws_norm'],
      dtype='object')

In [58]:
selected_cols = ['url_issued','PWS ID','PWS_clean','PWS Name',
                 'issued_date_issued','rescinded_date_rescinded','Population Served Count', 
                 'city_cleaned_issued','city_norm','Cities Served','city', 'Counties Served',
                 'county']

exact_matches = (
    exact_matches[selected_cols]
)
exact_matches.head(10)

,url_issued,PWS ID,PWS_clean,PWS Name,issued_date_issued,rescinded_date_rescinded,Population Served Count,city_cleaned_issued,city_norm,Cities Served,city,Counties Served,county
0,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2013913,quenemo,"QUENEMO, CITY OF",2024-02-28,2024-03-15,287,quenemo,quenemo,QUENEMO,Quenemo,"Osage, Osage",Osage County
1,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2013913,quenemo,"QUENEMO, CITY OF",2024-03-05,2024-03-15,287,quenemo,quenemo,QUENEMO,Quenemo,"Osage, Osage",Osage County
2,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2007304,fall river,"FALL RIVER, CITY OF",2024-03-22,2024-03-27,129,fall river,fall river,FALL RIVER,Fall River,"Greenwood, Greenwood",Greenwood County
3,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2011509,peabody,"PEABODY, CITY OF",2024-03-26,2024-03-28,932,peabody,peabody,PEABODY,Peabody,"Marion, Marion",Marion County
4,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2005304,kanopolis,"KANOPOLIS, CITY OF",2024-03-27,2024-03-30,453,kanopolis,kanopolis,KANOPOLIS,Kanopolis,"Ellsworth, Ellsworth",Ellsworth County
5,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2011509,peabody,"PEABODY, CITY OF",2024-03-28,2024-03-29,932,peabody,peabody,PEABODY,Peabody,"Marion, Marion",Marion County
6,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2000916,pawnee rock,"PAWNEE ROCK, CITY OF",2024-04-04,2024-04-05,190,pawnee rock,pawnee rock,PAWNEE ROCK,Pawnee Rock,"Barton, Barton",Barton County
7,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2013913,quenemo,"QUENEMO, CITY OF",2024-04-08,2024-04-16,287,quenemo,quenemo,QUENEMO,Quenemo,"Osage, Osage",Osage County
8,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2015514,haven,"HAVEN, CITY OF",2024-04-08,2024-04-10,1149,haven,haven,HAVEN,Haven,"Reno, Reno",Reno County
9,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2018302,kensington,"KENSINGTON, CITY OF",2024-04-09,2024-04-13,401,kensington,kensington,KENSINGTON,Kensington,"Smith, Smith",Smith County


In [59]:
matched_set = set(exact_matches["city_norm"])

unmatched_cities = (
    issued_rescinded_merged_df["city_norm"]
    .drop_duplicates()
    .loc[lambda x: ~x.isin(matched_set)]
)
unmatched_cities

1                   rolling hills landowners assoc
11                 western acres mobile home court
22                  tuttle cc mobile home park llc
40                    perry place mobile home park
49                                   geary co wd 2
58                                   sundowner inc
72      smith co rural water district 1   smith co
93                              midwest camps  llc
115                                 sundowner  inc
124    sunset valley mobile home park hays ks  llc
132                                        pwwsd 8
136                              eberly farms  inc
147                tuttle cc mobile home park  llc
149       sunset valley mobile home park  hays  ks
161                                          south
239                 neighbors of walnut grove  llc
273             colonial gardens mobile home court
279                                         coffey
281                                        woodson
Name: city_norm, dtype: object

In [60]:
from rapidfuzz import process, fuzz

pws_names = pws_df["pws_norm"].tolist()

fuzzy_results = []

for city in unmatched_cities:

    match, score, _ = process.extractOne(
        city,
        pws_names,
        scorer=fuzz.token_sort_ratio
    )

    fuzzy_results.append({
        "city_norm": city,
        "matched_pws": match,
        "score": score
    })

fuzzy_df = pd.DataFrame(fuzzy_results)

In [61]:
fuzzy_matches = fuzzy_df[fuzzy_df["score"] >= 90]

logging.info(f"Fuzzy matches: {len(fuzzy_matches)}")

[Info] Fuzzy matches: 8


In [62]:
fuzzy_matches

,city_norm,matched_pws,score
0,rolling hills landowners assoc,rolling hills landowners assn,94.915254
2,tuttle cc mobile home park llc,tuttle cc mobile home park,92.857143
9,sunset valley mobile home park hays ks llc,sunset valley mobile home park hays ks,95.000000
12,tuttle cc mobile home park llc,tuttle cc mobile home park,92.857143
13,sunset valley mobile home park hays ks,sunset valley mobile home park hays ks,100.000000
15,neighbors of walnut grove llc,neighbors of walnut grove,92.592593
16,colonial gardens mobile home court,colonial gardens mobile home ct,95.384615
18,woodson,woodston,93.333333


In [63]:
fuzzy_matches_full = fuzzy_matches.merge(
    pws_df,
    left_on="matched_pws",
    right_on="pws_norm",
    how="left"
)

In [64]:
fuzzy_matches_full.columns

Index(['city_norm', 'matched_pws', 'score', 'PWS ID', 'PWS Name',
       'EPA Region Name', 'Primacy Agency', 'PWS Type',
       'Population Served Count', 'Cities Served', 'Counties Served',
       '# of Facilities', '# of Violations', '# of Site Visits', 'PWS_clean',
       'pws_norm'],
      dtype='object')

In [65]:
fuzzy_matches_full = fuzzy_matches_full.merge(
    issued_rescinded_merged_df,
    on="city_norm",
    how="left"
)

In [66]:
fuzzy_matches_full.columns

Index(['city_norm', 'matched_pws', 'score', 'PWS ID', 'PWS Name',
       'EPA Region Name', 'Primacy Agency', 'PWS Type',
       'Population Served Count', 'Cities Served', 'Counties Served',
       '# of Facilities', '# of Violations', '# of Site Visits', 'PWS_clean',
       'pws_norm', 'url_issued', 'city', 'county', 'title_issued',
       'combined_context_issued', 'extracted_entities_advisory_reason_issued',
       'posted_on_issued', 'issued_date_issued', 'year_issued',
       'city_cleaned_issued', 'url_rescinded', 'rescinded_date_rescinded',
       'year_rescinded', 'duration_days'],
      dtype='object')

In [67]:
selected_cols = ['url_issued','PWS ID','PWS_clean','PWS Name',
                 'issued_date_issued','rescinded_date_rescinded','Population Served Count', 
                 'city_cleaned_issued','city_norm','Cities Served','city', 'Counties Served',
                 'county']
fuzzy_matches_full = fuzzy_matches_full[selected_cols]

In [68]:
fuzzy_matches_full.shape

(9, 13)

In [69]:
fuzzy_matches_full.head(14)

,url_issued,PWS ID,PWS_clean,PWS Name,issued_date_issued,rescinded_date_rescinded,Population Served Count,city_cleaned_issued,city_norm,Cities Served,city,Counties Served,county
0,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2017505,rolling hills landowners assn,ROLLING HILLS LANDOWNERS ASSN,2024-02-21,2024-04-18,130,rolling hills landowners assoc,rolling hills landowners assoc,LIBERAL,Rolling Hills Landowners Association,"Seward, Seward",Seward County
1,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2017505,rolling hills landowners assn,ROLLING HILLS LANDOWNERS ASSN,2024-04-10,2024-04-18,130,rolling hills landowners assoc,rolling hills landowners assoc,LIBERAL,Rolling Hills Landowners Association,"Seward, Seward",Seward County
2,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2016102,tuttle cc mobile home park,"TUTTLE CC MOBILE HOME PARK, LLC",2024-06-10,2024-06-19,50,tuttle cc mhp llc,tuttle cc mobile home park llc,MANHATTAN,Tuttle CC Mobile Home Park LLC,"Riley, Riley",Riley County
3,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2005101,sunset valley mhp hays ks,"SUNSET VALLEY MHP HAYS KS, LLC",2025-07-24,2025-07-30,60,"sunset valley mhp hays ks, llc",sunset valley mobile home park hays ks llc,HAYS,"Sunset Valley MHP Hays KS, LLC","Ellis, Ellis",Ellis County
4,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2016102,tuttle cc mobile home park,"TUTTLE CC MOBILE HOME PARK, LLC",2025-11-25,2025-12-04,50,"tuttle cc mhp, llc",tuttle cc mobile home park llc,MANHATTAN,"Tuttle CC Mobile Home Park, LLC","Riley, Riley",Riley County
5,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2005101,sunset valley mhp hays ks,"SUNSET VALLEY MHP HAYS KS, LLC",2025-12-03,2025-12-06,60,"sunset valley mhp, hays, ks",sunset valley mobile home park hays ks,HAYS,"Sunset Valley MHP, Hays, KS","Ellis, Ellis",Ellis County
6,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2014923,neighbors of walnut grove,"NEIGHBORS OF WALNUT GROVE, LLC",2023-07-12,2023-07-20,256,"neighbors of walnut grove, llc",neighbors of walnut grove llc,MANHATTAN,"Neighbors of Walnut Grove, LLC","Pottawatomie, Pottawatomie",Pottawatomie County
7,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2016118,colonial gardens mobile home ct,COLONIAL GARDENS MOBILE HOME CT,2023-10-24,2023-10-26,1200,colonial gardens mhc,colonial gardens mobile home court,MANHATTAN,Colonial Gardens Mobile Home Court,"Riley, Riley",Riley County
8,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2016307,woodston,"WOODSTON, CITY OF",2023-11-08,2023-11-13,95,woodson,woodson,WOODSTON,Woodson,"Rooks, Rooks",Coffey County


In [70]:
final_matches = pd.concat(
    [exact_matches, fuzzy_matches_full],
    ignore_index=True
)

In [71]:
final_matches

,url_issued,PWS ID,PWS_clean,PWS Name,issued_date_issued,rescinded_date_rescinded,Population Served Count,city_cleaned_issued,city_norm,Cities Served,city,Counties Served,county
0,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2013913,quenemo,"QUENEMO, CITY OF",2024-02-28,2024-03-15,287,quenemo,quenemo,QUENEMO,Quenemo,"Osage, Osage",Osage County
1,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2013913,quenemo,"QUENEMO, CITY OF",2024-03-05,2024-03-15,287,quenemo,quenemo,QUENEMO,Quenemo,"Osage, Osage",Osage County
2,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2007304,fall river,"FALL RIVER, CITY OF",2024-03-22,2024-03-27,129,fall river,fall river,FALL RIVER,Fall River,"Greenwood, Greenwood",Greenwood County
3,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2011509,peabody,"PEABODY, CITY OF",2024-03-26,2024-03-28,932,peabody,peabody,PEABODY,Peabody,"Marion, Marion",Marion County
4,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2005304,kanopolis,"KANOPOLIS, CITY OF",2024-03-27,2024-03-30,453,kanopolis,kanopolis,KANOPOLIS,Kanopolis,"Ellsworth, Ellsworth",Ellsworth County
...,...,...,...,...,...,...,...,...,...,...,...,...,...
262,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2016102,tuttle cc mobile home park,"TUTTLE CC MOBILE HOME PARK, LLC",2025-11-25,2025-12-04,50,"tuttle cc mhp, llc",tuttle cc mobile home park llc,MANHATTAN,"Tuttle CC Mobile Home Park, LLC","Riley, Riley",Riley County
263,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2005101,sunset valley mhp hays ks,"SUNSET VALLEY MHP HAYS KS, LLC",2025-12-03,2025-12-06,60,"sunset valley mhp, hays, ks",sunset valley mobile home park hays ks,HAYS,"Sunset Valley MHP, Hays, KS","Ellis, Ellis",Ellis County
264,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2014923,neighbors of walnut grove,"NEIGHBORS OF WALNUT GROVE, LLC",2023-07-12,2023-07-20,256,"neighbors of walnut grove, llc",neighbors of walnut grove llc,MANHATTAN,"Neighbors of Walnut Grove, LLC","Pottawatomie, Pottawatomie",Pottawatomie County
265,https://www.kdhe.ks.gov/m/newsflash/Home/Detai...,KS2016118,colonial gardens mobile home ct,COLONIAL GARDENS MOBILE HOME CT,2023-10-24,2023-10-26,1200,colonial gardens mhc,colonial gardens mobile home court,MANHATTAN,Colonial Gardens Mobile Home Court,"Riley, Riley",Riley County


In [72]:
final_matches.to_excel("final_for_geospatial.xlsx", index=False)